In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [2]:
feat = pd.read_csv('/Users/veron/Desktop/predicting_credit_card_payment/notebooks/credit_card_featured.csv')

In [3]:
raw_cols = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
payment_ratio_cols = ['PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY']

In [4]:
X = feat
y = X['default.payment.next.month']
X = feat[raw_cols + payment_ratio_cols]

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

In [6]:
logreg = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0, add_indicator=True)),
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

In [8]:
model = XGBClassifier(eval_metric = 'logloss', random_state=0)

In [9]:
for name, m in [('logreg', logreg), ('xgb', model)]:
    probs = cross_val_predict(m, X_train, y_train, cv=cv, method='predict_proba')[:, 1]


In [10]:
print(probs)

[0.01162645 0.01033933 0.11446184 ... 0.115658   0.53880477 0.61907744]


In [11]:
tn, fp, fn, tp = confusion_matrix(y_train, (probs >= 0.5).astype(int)).ravel()
print(tn, fp, fn, tp)

17553 1138 3371 1938


In [12]:
COST_FP = 50

In [13]:
def sweep(cost_fn):
    COST_FN = cost_fn
    rows = []
    for t in np.arange(0.01, 0.96, 0.01):
        tn, fp, fn, tp = confusion_matrix(y_train, (probs >= t).astype(int)).ravel()
        rows.append({
            'threshold': t,
            'flagged': tp + fp,
            'fn': fn,
            'fp': fp,
            'cost': fn * COST_FN + fp * COST_FP,
            'recall': tp / (tp + fn),
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0
        })

    df_sweep = pd.DataFrame(rows)

    best = df_sweep.loc[df_sweep['cost'].idxmin()]
    return df_sweep, best

In [14]:
table, best = sweep(1000)
print(best)


threshold         0.010000
flagged       23103.000000
fn               40.000000
fp            17834.000000
cost         931700.000000
recall            0.992466
precision         0.228066
Name: 0, dtype: float64


In [15]:
n_train = len(y_train)
cap_rows = []
for pct in [0.05, 0.10, 0.20]:
    target = pct * n_train
    row = table.loc[(table['flagged'] - target).abs().idxmin()]
    cap_rows.append({
        'capacity_pct': pct,
        'threshold': round(row['threshold'], 2),
        'flagged': int(row['flagged']),
        'recall': round(row['recall'], 3),
        'precision': round(row['precision'], 3),
    })

capacity_table = pd.DataFrame(cap_rows)
print(capacity_table)

   capacity_pct  threshold  flagged  recall  precision
0          0.05       0.77     1204   0.165      0.726
1          0.10       0.59     2394   0.301      0.667
2          0.20       0.35     4715   0.478      0.538


Under the assumed cost ratio of 20:1, the cost-minimizing threshold falls below 0.01, flagging nearly the entire portfolio. This is a property of the assumption, not a useful policy: when a miss is 20x a false alarm and 22% of customers default, blanket intervention beats any selective strategy. Only at a 5:1 ratio does a selective threshold (0.15) emerge.

Contacting the model's top-ranked 10% of customers identifies 30% of all defaulters — three times what random selection would achieve — with two-thirds of contacts reaching a customer who does go on to default.